# 02. Extended majority-vote upward aggregation (Table A.15)

The main pipeline uses a *forward-fill* baseline for cross-frequency
comparison: coarse labels are broadcast down to the fine 5m grid. This
notebook walks through the *symmetric* counterpart used in Table A.15:
fine labels are aggregated *upward* into coarse-grid bins via majority
vote. We run it in-process on `CL`, then display the cached
all-asset summary.

**Re-run command**: `python run.py extended_majority_vote`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- load raw 5m OHLC for `CL`

Calls `src.data.data_ib.load_5m_ohlc` directly on `data/CL_5m.csv`. This is the same loader the pipeline uses; it parses mixed-offset DST timestamps as UTC then converts to NY local time.

In [2]:
from src.data.data_ib import load_5m_ohlc

FOCAL = "CL"
df_5m = load_5m_ohlc(DATA / f"{FOCAL}_5m.csv")
print(f"{FOCAL}_5m: {len(df_5m):,} bars, "
      f"{df_5m.index.min()} -> {df_5m.index.max()}, "
      f"{df_5m.index.normalize().nunique()} trading days")
df_5m.head()

CL_5m: 34,896 bars, 2025-11-02 18:00:00-05:00 -> 2026-05-01 16:55:00-04:00, 155 trading days


,Open,High,Low,Close,Volume
Date,,,,,
2025-11-02 18:00:00-05:00,58.36,58.43,58.10,58.29,2872.0
2025-11-02 18:05:00-05:00,58.30,58.37,58.29,58.35,669.0
2025-11-02 18:10:00-05:00,58.35,58.36,58.29,58.30,367.0
2025-11-02 18:15:00-05:00,58.30,58.33,58.26,58.33,394.0
2025-11-02 18:20:00-05:00,58.33,58.33,58.30,58.31,313.0


## Step 2 -- compute majority-vote ARI in-process

`src.experiments.exp_01_majority_vote.majority_vote_ari` fits GMM regimes
independently at every frequency, then for each pair (fine, coarse) bins
the fine labels into coarse-sized intervals (e.g. fifteen 5m bars per 15m
bar) and labels each bin by majority vote (ties to crisis). Compares the
aggregated stream to the natively-coarse stream via ARI.

In [ ]:
from src.experiments.exp_01_majority_vote import majority_vote_ari
from src.workflows.pipeline import mean_offdiag_ari

mv_ari = majority_vote_ari(df_5m, FOCAL)
print(f"mean off-diag ARI (majority-vote, in-process): {mean_offdiag_ari(mv_ari):.4f}")
mv_ari.round(3)

## Step 3 -- cross-check against cached per-asset CSV

In [4]:
p = OUT / f"CL_majority_vote_cross_freq_ari.csv"
if p.exists():
    cached = pd.read_csv(p, index_col=0)
    display(Markdown(f"### Cached `outputs/CL_majority_vote_cross_freq_ari.csv`"))
    display(cached.round(3))
    diff = (mv_ari - cached).abs().max().max()
    print(f"max abs diff (in-process vs cached): {diff:.6f}")
else:
    display(Markdown(f"`{p}` missing -- in-process result above is the only available number."))

### Cached `outputs/CL_majority_vote_cross_freq_ari.csv`

,5m,15m,1h,1d
5m,1.000,0.697,0.363,-0.049
15m,0.697,1.000,0.340,-0.067
1h,0.363,0.340,1.000,-0.028
1d,-0.049,-0.067,-0.028,1.000


max abs diff (in-process vs cached): 0.000000


## Cached all-asset summary -- `outputs/majority_vote_summary.csv`

In [5]:
p = OUT / "majority_vote_summary.csv"
display(pd.read_csv(p).round(3) if p.exists() else Markdown(f"`{p}` missing"))

,symbol,mean_offdiag_ari_majority_vote
0,SPY,0.223
1,USDJPY,0.094
2,CL,0.209
3,GLD,0.135


## Cached per-asset matrices

In [6]:
for asset in ['SPY', 'USDJPY', 'CL', 'GLD']:
    p = OUT / f"{asset}_majority_vote_cross_freq_ari.csv".format(asset=asset)
    if not p.exists():
        continue
    display(Markdown(f"### {asset}"))
    display(pd.read_csv(p, index_col=0).round(3))

### SPY

,5m,15m,1h,1d
5m,1.000,0.539,0.136,0.169
15m,0.539,1.000,0.151,0.157
1h,0.136,0.151,1.000,0.184
1d,0.169,0.157,0.184,1.000


### USDJPY

,5m,15m,1h,1d
5m,1.000,0.412,0.118,-0.056
15m,0.412,1.000,0.180,-0.035
1h,0.118,0.180,1.000,-0.055
1d,-0.056,-0.035,-0.055,1.000


### CL

,5m,15m,1h,1d
5m,1.000,0.697,0.363,-0.049
15m,0.697,1.000,0.340,-0.067
1h,0.363,0.340,1.000,-0.028
1d,-0.049,-0.067,-0.028,1.000


### GLD

,5m,15m,1h,1d
5m,1.000,0.704,0.144,-0.075
15m,0.704,1.000,0.138,-0.072
1h,0.144,0.138,1.000,-0.031
1d,-0.075,-0.072,-0.031,1.000
